# Day 14 – Prompt Chaining & Self‑Consistency – Extensive Solutions

Complete implementations of multi‑step chains, self‑consistency for reasoning, and conditional branching.

In [ ]:
import openai
from dotenv import load_dotenv
import os
from collections import Counter
import re

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def ask(prompt, temp=0, max_tokens=500):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print("Ready.")

## Exercise 1: Customer support chain (classify → retrieve → personalise)

We'll build a three‑step chain for handling customer complaints.

In [ ]:
# Step 1: Classify intent
def classify_intent(message):
    prompt = f"Classify the intent of this customer message as: Billing, Technical, or General. Message: {message}"
    return ask(prompt, temp=0).strip()

# Step 2: Retrieve policy (simulated database)
policy_db = {
    "Billing": "Our refund policy allows returns within 30 days.",
    "Technical": "Please try restarting your device and clearing cache.",
    "General": "Thank you for contacting support. How can we help?"
}

# Step 3: Personalise response
def personalise_response(intent, policy, original_message):
    prompt = f"""Write a polite, empathetic response to the customer.
    Intent: {intent}
    Policy: {policy}
    Customer message: {original_message}
    Response:"""
    return ask(prompt, temp=0.5)

# Test the chain
customer_msg = "I was charged twice for my subscription last month!"
intent = classify_intent(customer_msg)
policy = policy_db.get(intent, policy_db["General"])
response = personalise_response(intent, policy, customer_msg)
print(f"Intent: {intent}\nPolicy: {policy}\n\nFinal response:\n{response}")

## Exercise 2: Self‑consistency on a logic puzzle

Puzzle: *"There are three boxes: one contains apples, one oranges, one both. All labels are wrong. You pick one fruit from one box. Determine contents."*

We'll run 5 reasoning chains and take majority answer.

In [ ]:
logic_puzzle = """
There are three boxes: one labeled 'Apples', one labeled 'Oranges', one labeled 'Apples and Oranges'.
All labels are incorrect. You are allowed to take one fruit from one box only.
How can you determine the correct labels? Explain step by step.
"""

answers = []
for i in range(5):
    reasoning = ask(logic_puzzle, temp=0.8, max_tokens=300)
    # Extract final conclusion (simple heuristic: look for key phrase)
    if "take from the box labeled 'Apples and Oranges'" in reasoning:
        answers.append("Take from 'Apples and Oranges'")
    elif "take from the box labeled 'Apples'" in reasoning:
        answers.append("Take from 'Apples'")
    else:
        answers.append("Other")
    print(f"Chain {i+1}: {reasoning[:100]}...\n")

majority = Counter(answers).most_common(1)[0][0]
print(f"Majority answer: {majority}")
print("Correct answer: Take from the box labeled 'Apples and Oranges' because that box must contain only one fruit, revealing the others.")

## Exercise 3: Chain with RAG

We'll integrate retrieval into a chain: retrieve relevant documents → summarise → answer a question.

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings

# Reuse vectorstore from Day 13 (or create a small one)
docs = ["The Eiffel Tower is 330 meters tall.", "The Eiffel Tower is in Paris."]
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
chunks = splitter.create_documents(docs)
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings())

def rag_chain(question):
    # Retrieve
    retrieved = vectorstore.similarity_search(question, k=2)
    context = "\n".join([d.page_content for d in retrieved])
    # Summarise context
    summary = ask(f"Summarise this in one sentence: {context}", temp=0)
    # Answer based on summary
    answer = ask(f"Based on: {summary}\nAnswer: {question}", temp=0)
    return answer

print(rag_chain("How tall is the Eiffel Tower?"))

## Exercise 4: Parallel chaining (generate multiple drafts, then select best)

Generate three product descriptions with different tones, then ask a second LLM to choose the best.

In [ ]:
product = "wireless noise-cancelling headphones"
tones = ["professional", "enthusiastic", "minimalist"]
drafts = []

for tone in tones:
    prompt = f"Write a {tone} product description for {product} in 2 sentences."
    draft = ask(prompt, temp=0.7)
    drafts.append(draft)
    print(f"--- {tone.capitalize()} draft ---\n{draft}\n")

# Second LLM as judge
judge_prompt = f"""
Which of the following product descriptions is the most effective for selling? Reply with just the number (1, 2, or 3).
1. {drafts[0]}
2. {drafts[1]}
3. {drafts[2]}
"""
best_idx = int(ask(judge_prompt, temp=0).strip()) - 1
print(f"\nBest description (draft {best_idx+1}):\n{drafts[best_idx]}")